This notebook seeks to replicate the pre-processing steps for EJSCREEN's traffic proximity indicator. The pre-processing was done in ArcGIS (see [here](https://github.com/USEPA-clone/ejscreen-traffic-proximity-processing/blob/main/README.md)). The notebook provides an open-source alternative appraoch.

In [ ]:
# Imports
import pandas, geopandas

In [ ]:
# Import HPMS traffic segments from ArcGIS feature server
import requests
import geopandas as gpd
import pandas as pd

def get_all_arcgis_features(base_url):
    query_url = f"{base_url}/query"

    # 1. Get all Object IDs first
    print("Fetching all Object IDs...")
    id_params = {
        'where': '1=1',
        'returnIdsOnly': 'true',
        'f': 'json'
    }
    id_response = requests.get(query_url, params=id_params)
    id_response.raise_for_status()
    all_ids = id_response.json().get('objectIds', [])

    total_records = len(all_ids)
    print(f"Total records found: {total_records}")

    # 2. Split IDs into chunks of 1000
    chunk_size = 1000
    id_chunks = [all_ids[i:i + chunk_size] for i in range(0, total_records, chunk_size)]

    all_features = []

    # 3. Fetch each chunk by ID
    for i, chunk in enumerate(id_chunks):
        # Convert ID list to comma-separated string
        ids_str = ",".join(map(str, chunk))

        params = {
            'objectIds': ids_str,
            'outFields': '*',
            'f': 'geojson',
            'returnGeometry': 'true'
        }

        print(f"Downloading chunk {i+1}/{len(id_chunks)}...")
        response = requests.get(query_url, params=params)
        response.raise_for_status()

        data = response.json()
        all_features.extend(data.get('features', []))

    # 4. Wrap into a GeoDataFrame
    feature_collection = {
        "type": "FeatureCollection",
        "features": all_features
    }

    gdf = gpd.GeoDataFrame.from_features(feature_collection)
    gdf.set_crs(epsg=4326, inplace=True)

    return gdf

# URL
url = "https://geo.dot.gov/server/rest/services/Hosted/HPMS_FULL_RI_2020/FeatureServer/0" # Change from HPMS_FULL_RI_2020 to some other state...

gdf_full = get_all_arcgis_features(url)
print(f"\nFinal count: {len(gdf_full)} records.")

Fetching all Object IDs...
Total records found: 41269

Final count: 41269 records.


In [ ]:
# Create subset of "Major" highway segments using functional class (f_system in (1, 2, 3) or (f_system = 4 and urban_code <> 99999)
gdf_full['f'] = gdf_full['f_system'].astype("Int64") # Convert data format to integer
gdf_full['u'] = gdf_full['urban_code'].astype("Int64") # Convert data format to integer
subset = gdf_full[(gdf_full['f'].isin([1,2,3])) | ((gdf_full["f"]==4) & (gdf_full['u'] != 99999))]
subset

,geometry,pct_peak_single,maintenance_operations,k_factor,route_id,future_aadt_year,end_point,sample_id,route_number,number_signals,...,ownership,nhs,strahnet_type,expansion_factor,surface_type,objectid,toll_id,shoulder_width_l,f,u
0,"LINESTRING (-71.54367 41.87784, -71.54358 41.8...",NaN,NaN,NaN,3900,NaN,14.800,None,116.0,NaN,...,1.0,1.0,NaN,NaN,6.0,1,None,NaN,3,72505
2,"LINESTRING (-71.38095 41.8859, -71.38082 41.88...",NaN,NaN,NaN,20400,NaN,0.418,None,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,3,None,NaN,4,72505
3,"LINESTRING (-71.36131 41.85427, -71.36155 41.8...",NaN,NaN,NaN,3700,NaN,28.500,None,114.0,NaN,...,1.0,1.0,NaN,NaN,6.0,4,None,NaN,3,72505
7,"LINESTRING (-71.35194 41.79555, -71.35198 41.7...",NaN,NaN,10.0,3700,NaN,24.174,None,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,8,None,NaN,4,72505
8,"LINESTRING (-71.72548 41.35262, -71.7253 41.35...",0.441,NaN,9.0,100,2.177453e+12,7.091,105379001000,NaN,NaN,...,NaN,1.0,NaN,1.353,NaN,9,None,NaN,3,99999
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41263,"LINESTRING (-71.78354 41.34044, -71.78344 41.3...",0.233,NaN,8.0,86000,2.177453e+12,6.700,136558010000,1.0,NaN,...,1.0,NaN,NaN,3.414,6.0,41264,None,NaN,4,64135
41264,"LINESTRING (-71.39166 41.76745, -71.39176 41.7...",0.262,NaN,9.0,600150,2.177453e+12,0.300,107449010000,NaN,NaN,...,4.0,NaN,NaN,6.765,6.0,41265,None,NaN,4,72505
41266,"LINESTRING (-71.55924 41.94519, -71.5592 41.94...",NaN,NaN,12.0,500,NaN,19.791,None,5.0,NaN,...,1.0,NaN,NaN,NaN,NaN,41267,None,NaN,4,72505
41267,"LINESTRING (-71.71445 41.81858, -71.71392 41.8...",0.275,NaN,8.0,600,2.177453e+12,4.525,112389013000,6.0,NaN,...,1.0,1.0,NaN,1.267,6.0,41268,None,NaN,3,99999


In [ ]:
# Remove records where AADT is NULL or AADT = 0 or Shape_length = 0
subset['a'] = subset['aadt'].astype("Int64") # Convert data format to integer
subset['s'] = pandas.to_numeric(subset['SHAPE__Length'], errors="coerce") # Convert data format
s2 = subset.query('a > 0') # Remove records where AADT = null or AADT = 0
s2 = s2.query('s > 0') # Remove records where shape length = 0
s2

/usr/local/lib/python3.12/dist-packages/geopandas/geodataframe.py:1969: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)
/usr/local/lib/python3.12/dist-packages/geopandas/geodataframe.py:1969: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)
/tmp/ipython-input-1173266060.py:7: RuntimeWarning: Engine has switched to 'python' because numexpr does not support extension array dtypes. Please set your engine to python manually.
  s2 = subset.query('a > 0')

,geometry,pct_peak_single,maintenance_operations,k_factor,route_id,future_aadt_year,end_point,sample_id,route_number,number_signals,...,strahnet_type,expansion_factor,surface_type,objectid,toll_id,shoulder_width_l,f,u,a,s
0,"LINESTRING (-71.54367 41.87784, -71.54358 41.8...",NaN,NaN,NaN,3900,NaN,14.800,None,116.0,NaN,...,NaN,NaN,6.0,1,None,NaN,3,72505,6866,0.001857
2,"LINESTRING (-71.38095 41.8859, -71.38082 41.88...",NaN,NaN,NaN,20400,NaN,0.418,None,NaN,NaN,...,NaN,NaN,NaN,3,None,NaN,4,72505,7208,0.001919
3,"LINESTRING (-71.36131 41.85427, -71.36155 41.8...",NaN,NaN,NaN,3700,NaN,28.500,None,114.0,NaN,...,NaN,NaN,6.0,4,None,NaN,3,72505,7203,0.001668
7,"LINESTRING (-71.35194 41.79555, -71.35198 41.7...",NaN,NaN,10.0,3700,NaN,24.174,None,NaN,NaN,...,NaN,NaN,NaN,8,None,NaN,4,72505,8801,0.012009
8,"LINESTRING (-71.72548 41.35262, -71.7253 41.35...",0.441,NaN,9.0,100,2.177453e+12,7.091,105379001000,NaN,NaN,...,NaN,1.353,NaN,9,None,NaN,3,99999,8308,0.006897
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41263,"LINESTRING (-71.78354 41.34044, -71.78344 41.3...",0.233,NaN,8.0,86000,2.177453e+12,6.700,136558010000,1.0,NaN,...,NaN,3.414,6.0,41264,None,NaN,4,64135,5837,0.001880
41264,"LINESTRING (-71.39166 41.76745, -71.39176 41.7...",0.262,NaN,9.0,600150,2.177453e+12,0.300,107449010000,NaN,NaN,...,NaN,6.765,6.0,41265,None,NaN,4,72505,8453,0.001216
41266,"LINESTRING (-71.55924 41.94519, -71.5592 41.94...",NaN,NaN,12.0,500,NaN,19.791,None,5.0,NaN,...,NaN,NaN,NaN,41267,None,NaN,4,72505,3091,0.001451
41267,"LINESTRING (-71.71445 41.81858, -71.71392 41.8...",0.275,NaN,8.0,600,2.177453e+12,4.525,112389013000,6.0,NaN,...,NaN,1.267,6.0,41268,None,NaN,3,99999,10270,0.001873


In [ ]:
# Dissolve redundant and partial overlapping segments
# First, add new column: REPEAT_TEST
# Calculate: REPEAT_TEST = Route_ID & AADT
s2["REPEAT_TEST"] = s2['route_id'].astype(str) + "-" + s2['a'].astype(str)
s2["REPEAT_TEST"]

,REPEAT_TEST
0,3900-6866
2,20400-7208
3,3700-7203
7,3700-8801
8,100-8308
...,...
41263,86000-5837
41264,600150-8453
41266,500-3091
41267,600-10270


In [ ]:
# Run ArcGIS Dissolve tool (dissolve by REPEAT_TEST; first­_state_code, first_urban_code, mean_AADT), "No Multipart", "No Unsplit: HPMS2020_1_clean, … HPMS2020_72_clean. Note switching StateAbb to StateFIPS here for state identifier. Note that state abbreviations are changed to FIPS codes for each file (for example, AL to 1, WY to 56, PR to 72)

agg = {"state_code": "first", "u": "first", "f": "first", "a": "mean"}
#final = s2.groupby(by="REPEAT_TEST").agg(agg)
final = s2.dissolve(by="REPEAT_TEST", aggfunc=agg)
final.reset_index(inplace=True)
final

,REPEAT_TEST,geometry,state_code,u,f,a
0,10-DEXTERRD-3-4159,"MULTILINESTRING ((-71.37019 41.82928, -71.3704...",44,72505,4,4159.0
1,10-WARRENAVE-3-1946,"MULTILINESTRING ((-71.38432 41.81832, -71.3842...",44,72505,4,1946.0
2,100-10202,"MULTILINESTRING ((-71.45013 41.66398, -71.4501...",44,72505,3,10202.0
3,100-10293,"MULTILINESTRING ((-71.3843 41.87714, -71.38413...",44,72505,3,10293.0
4,100-10467,"MULTILINESTRING ((-71.41281 41.83152, -71.4122...",44,72505,3,10467.0
...,...,...,...,...,...,...
2319,999600-12840,"MULTILINESTRING ((-71.49904 41.82391, -71.4968...",44,72505,4,12840.0
2320,999600-6623,"MULTILINESTRING ((-71.47821 41.82022, -71.4781...",44,72505,4,6623.0
2321,999600-7421,"MULTILINESTRING ((-71.44555 41.81685, -71.4455...",44,72505,4,7421.0
2322,999600-8400,"MULTILINESTRING ((-71.44567 41.81686, -71.4455...",44,72505,4,8400.0


In [ ]:
# Add ID (long int, calculated from OBJECTID)
final['ID'] = final.index
final

,REPEAT_TEST,geometry,state_code,u,f,a,ID
0,10-DEXTERRD-3-4159,"MULTILINESTRING ((-71.37019 41.82928, -71.3704...",44,72505,4,4159.0,0
1,10-WARRENAVE-3-1946,"MULTILINESTRING ((-71.38432 41.81832, -71.3842...",44,72505,4,1946.0,1
2,100-10202,"MULTILINESTRING ((-71.45013 41.66398, -71.4501...",44,72505,3,10202.0,2
3,100-10293,"MULTILINESTRING ((-71.3843 41.87714, -71.38413...",44,72505,3,10293.0,3
4,100-10467,"MULTILINESTRING ((-71.41281 41.83152, -71.4122...",44,72505,3,10467.0,4
...,...,...,...,...,...,...,...
2319,999600-12840,"MULTILINESTRING ((-71.49904 41.82391, -71.4968...",44,72505,4,12840.0,2319
2320,999600-6623,"MULTILINESTRING ((-71.47821 41.82022, -71.4781...",44,72505,4,6623.0,2320
2321,999600-7421,"MULTILINESTRING ((-71.44555 41.81685, -71.4455...",44,72505,4,7421.0,2321
2322,999600-8400,"MULTILINESTRING ((-71.44567 41.81686, -71.4455...",44,72505,4,8400.0,2322


In [ ]:
# Map
import folium

m = folium.Map(location=[41.5800, -71.4774], zoom_start=10)

folium.GeoJson(
  final
).add_to(m)

m

In [ ]:
# Compare the results of this approach to those from the ArcGIS-based approach
## Load pre-processed JSON created in ArcGIS

arcgis = geopandas.read_file("https://raw.githubusercontent.com/Public-Environmental-Data-Partners/EJSCREEN-Data-Processing/refs/heads/main/outputs/traffic/preprocessing/HPMS2020_44.json")

print(arcgis['aadt'].describe()) # ArcGIS summary stats
print(final['a'].describe()) # This approach summary stats

count      2359.000000
mean      13868.569309
std       19748.900333
min          87.000000
25%        5036.000000
50%        8939.000000
75%       14005.500000
max      173724.000000
Name: aadt, dtype: float64
count          2324.0
mean     13935.187608
std      19881.053102
min              87.0
25%            5023.0
50%            8931.0
75%          14098.75
max          173724.0
Name: a, dtype: Float64


In [ ]:
# Use ArcGIS "export to geodatabase" tool to convert Polyline M to Polyline features and alter table structure ready for JSON export, including renaming fields: state_code, f_system, urban_code, aadt, ID -- HPMS2020_1_forJSON, … HPMS2020_72_forJSON

# Above is not applicable here per se (though would need to adjust the processing scripts to handle a plain JSON file rather than Esri JSON)

In [ ]:
# Export to JSON using ESRI Hadoop tool: ../InputforHadoop/HPMS2020_1.json, …, HPMS2020_72.json. Note that toolbox is provided by Esri geoprocessing-tools-for-hadoop-master/HadoopTools.py (use Features to JSON with default settings)
# Resulting JSON fie elements should be state_code, f_system, urban_code, aadt, ID, geometry (z and m both false)

# Above is not applicable here per se (though would need to adjust the processing scripts to handle a plain JSON file rather than Esri JSON)